# Phase 1: Environment and Model Setup — Sparse Autoencoders on GPT-2 Small

**Project:** Sparse Autoencoders on GPT-2 Small  
**Target Model:** Pretrained GPT-2 Small (`gpt2-small`, $d_{model} = 768$, 12 layers)  
**Target Residual Stream Hook Site:** `blocks.6.hook_resid_post` (Layer 6 residual stream)  
**Target Corpus:** WikiText-2  
**Execution Support:** Google Colab (CPU & GPU) and Local Environments  

---

## 1. Overview & Research Goal
This research project investigates the central research question:
> **How does changing the sparsity penalty of a sparse autoencoder affect its ability to reconstruct activations from GPT-2 Small, the number and frequency of active latents, and the contexts that activate learned features?**

### Phase 1 Objectives:
1. **Environment Verification:** Verify Python and core library versions (`torch`, `transformer_lens`, `transformers`, `datasets`, `einops`).
2. **Hardware Agnosticism:** Seamlessly run on CPU by default and leverage CUDA GPU if available.
3. **Model Loading & Parameter Immutability:** Load pretrained `gpt2-small` via TransformerLens, set `model.eval()`, and freeze 100% of parameters (`requires_grad = False`).
4. **Forward Pass Verification:** Inspect tokenization and output logit tensor shapes.
5. **Activation Extraction at Documented Hook:** Extract intermediate activations at `blocks.6.hook_resid_post` using `torch.inference_mode()`.
6. **Formal Assertions:** Programmatically assert that all parameters are frozen and extracted activations require no gradients.
7. **Persistent CPU-Compatible Storage:** Serialize a sample activation tensor and structured JSON metadata to `checkpoints/`.
8. **Colab Checkpointing Guide:** Provide Google Drive mount and browser download functions to prevent data loss.

## 2. Package Installation & Dependency Compatibility

> ### ⚠️ API Compatibility Notice: TransformerLens & Transformers
> In `TransformerLens 4.0+`, the foundational `HookedTransformer` class was removed and replaced with `TransformerBridge`.
> To maintain compatibility with standard mechanistic interpretability workflows and official SAE Lens patterns,
> we pin `transformer-lens>=2.0.0,<3.0.0` and `transformers>=4.40.0,<5.0.0`.

In [ ]:
import sys

# Check if running inside Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Detected Google Colab environment. Installing pinned dependencies...")
    !pip install -q "transformer-lens>=2.0.0,<3.0.0" "transformers>=4.40.0,<5.0.0" "datasets>=2.15.0" "einops>=0.7.0" "jaxtyping>=0.2.25" "safetensors>=0.4.0"
    print("Colab dependencies installed successfully.")
else:
    print("Detected local development environment. Using local virtual environment packages.")

## 3. Environment Inspection & Deterministic Seed Setup

We record system information, package versions, and enforce deterministic random seeds across PyTorch and NumPy.

In [ ]:
import os
import json
import random
import importlib.metadata
from datetime import datetime, timezone

import numpy as np
import torch
import transformers
import datasets
import transformer_lens
from transformer_lens import HookedTransformer

# Set deterministic seeds
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Determine compute device
device = "cuda" if torch.cuda.is_available() else "cpu"

def get_version(pkg_name):
    try:
        return importlib.metadata.version(pkg_name)
    except importlib.metadata.PackageNotFoundError:
        return "Not installed"

print("=" * 65)
print("ENVIRONMENT & DEPENDENCY STATUS")
print("=" * 65)
print(f"Python Version         : {sys.version.split()[0]}")
print(f"PyTorch Version        : {torch.__version__}")
print(f"TransformerLens Version: {get_version('transformer-lens')}")
print(f"Transformers Version   : {transformers.__version__}")
print(f"Datasets Version       : {datasets.__version__}")
print(f"Einops Version         : {get_version('einops')}")
print(f"Target Execution Device: {device}")
if device == "cuda":
    print(f"CUDA Device Name       : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Memory Allocated  : {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
print(f"Deterministic Seed     : {RANDOM_SEED}")
print("=" * 65)

## 4. Loading Pretrained GPT-2 Small & Freezing Model Parameters

In sparse autoencoder research, the language model serves strictly as an immutable activation generator:
- We **never backpropagate** into the base model.
- Setting `model.eval()` turns off dropout and nondeterministic layers.
- Setting `param.requires_grad = False` prevents PyTorch from allocating gradient memory.

In [ ]:
MODEL_ID = "gpt2-small"

print(f"Loading pretrained '{MODEL_ID}' onto device '{device}'...")
model = HookedTransformer.from_pretrained(MODEL_ID, device=device)

# 1. Set to evaluation mode
model.eval()

# 2. Explicitly freeze all model parameters
for p in model.parameters():
    p.requires_grad = False

print(f"Successfully loaded '{MODEL_ID}'.")
print(f"Architecture Config:")
print(f" - Layers (n_layers)     : {model.cfg.n_layers}")
print(f" - Hidden Dim (d_model)  : {model.cfg.d_model}")
print(f" - Attention Heads       : {model.cfg.n_heads} (head dim: {model.cfg.d_head})")
print(f" - MLP Dimension (d_mlp) : {model.cfg.d_mlp}")
print(f" - Vocabulary Size       : {model.cfg.d_vocab}")
print(f" - Context Length (n_ctx): {model.cfg.n_ctx}")

## 5. Verification Check: Rigorous Parameter Audit

We loop through all parameter tensors in `model.named_parameters()` to verify that zero parameters have `requires_grad=True`.

In [ ]:
total_params = 0
trainable_params = 0
unfrozen_tensors = []

for name, param in model.named_parameters():
    numel = param.numel()
    total_params += numel
    if param.requires_grad:
        trainable_params += numel
        unfrozen_tensors.append(name)

# Strict assertions
assert len(unfrozen_tensors) == 0, f"Audit Failed! Found unfrozen parameters: {unfrozen_tensors}"
assert trainable_params == 0, f"Audit Failed! {trainable_params} parameters require gradients!"
assert not model.training, "Audit Failed! Model is in training mode, expected eval mode."

print("CHECK PASSED: Parameter Freezing Audit")
print(f"Total Model Parameters  : {total_params:,}")
print(f"Trainable Parameters    : {trainable_params} (0.00%)")
print(f"Model Training Flag     : {model.training} (eval mode verified)")

## 6. Forward Pass & Logit Shape Inspection

We pass a sample sentence through the model to confirm tokenization mechanics and inspect output logit dimensions `[batch_size, seq_len, d_vocab]`.

In [ ]:
sample_sentence = "Sparse autoencoders discover interpretable features in language models."
tokens = model.to_tokens(sample_sentence)

with torch.inference_mode():
    logits = model(tokens)

print(f"Sample Input String : '{sample_sentence}'")
print(f"Token IDs Tensor    : {tokens}")
print(f"Token String Split  : {model.to_str_tokens(sample_sentence)}")
print(f"Tokens Shape        : {tokens.shape} -> [batch={tokens.shape[0]}, seq_len={tokens.shape[1]}]")
print(f"Logits Tensor Shape : {logits.shape} -> [batch={logits.shape[0]}, seq_len={logits.shape[1]}, d_vocab={logits.shape[2]}]")

next_token_id = logits[0, -1].argmax().item()
print(f"Greedy Next Token   : '{model.to_string([next_token_id])}' (ID: {next_token_id})")

## 7. Activation Extraction at Documented Hook Site

For our SAE investigation, we target **Layer 6 residual stream** (`blocks.6.hook_resid_post`).
- Layer 6 is the exact midpoint of GPT-2 Small (12 layers: 0 to 11).
- Mid-layer residual stream representations exhibit rich semantic and syntactic superposition, making them the primary site for dictionary learning in mechanistic interpretability literature.

In [ ]:
HOOK_NAME = "blocks.6.hook_resid_post"

# Extract activations with run_with_cache inside inference_mode
with torch.inference_mode():
    logits, cache = model.run_with_cache(tokens)

assert HOOK_NAME in cache, f"Target hook '{HOOK_NAME}' not found in cache! Available keys: {list(cache.keys())[:10]}"
activation = cache[HOOK_NAME]

print("=" * 65)
print("HOOK ACTIVATION EXTRACTION SUMMARY")
print("=" * 65)
print(f"Model Identifier   : {MODEL_ID}")
print(f"Compute Device     : {device}")
print(f"Random Seed        : {RANDOM_SEED}")
print(f"Target Hook Site   : {HOOK_NAME}")
print(f"Activation Shape   : {list(activation.shape)} -> [batch, seq_len, d_model]")
print(f"Activation Dtype   : {activation.dtype}")
print(f"Mean Value         : {activation.mean().item():.6f}")
print(f"Standard Deviation : {activation.std().item():.6f}")
print("=" * 65)

## 8. Verification Check: Gradient-Free Activations

We verify that both the output logits and intermediate activations do not track gradients.

In [ ]:
assert not logits.requires_grad, "Audit Failed! Logits require gradients."
assert logits.grad_fn is None, "Audit Failed! Logits have an active grad_fn."
assert not activation.requires_grad, f"Audit Failed! Hook activation {HOOK_NAME} requires gradients."
assert activation.grad_fn is None, f"Audit Failed! Hook activation {HOOK_NAME} has an active grad_fn."

print("CHECK PASSED: Extracted activations and logits are strictly gradient-free.")

## 9. CPU-Compatible Activation Cache & Structured Metadata

Activations must be persisted to disk in a format that:
1. Is completely portable to CPU-only environments (`tensor.detach().cpu()`).
2. Includes structured JSON metadata to record provenance, model ID, hook point, and token contexts.

In [ ]:
OUTPUT_DIR = os.path.join("checkpoints", "sample_cache")
os.makedirs(OUTPUT_DIR, exist_ok=True)

cache_tensor_path = os.path.join(OUTPUT_DIR, "activation_blocks_6_resid_post.pt")
metadata_path = os.path.join(OUTPUT_DIR, "activation_blocks_6_resid_post_metadata.json")

# Ensure CPU tensor for cross-platform compatibility
activation_cpu = activation.detach().cpu()
torch.save(activation_cpu, cache_tensor_path)

metadata = {
    "model_identifier": MODEL_ID,
    "hook_name": HOOK_NAME,
    "tensor_shape": list(activation_cpu.shape),
    "tensor_dtype": str(activation_cpu.dtype),
    "execution_device": device,
    "random_seed": RANDOM_SEED,
    "sample_prompt": sample_sentence,
    "token_strings": model.to_str_tokens(sample_sentence),
    "num_tokens": tokens.shape[1],
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "package_versions": {
        "torch": torch.__version__,
        "transformer_lens": get_version("transformer-lens"),
        "transformers": transformers.__version__,
        "datasets": datasets.__version__
    }
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved Activation Tensor : {cache_tensor_path} ({os.path.getsize(cache_tensor_path)} bytes)")
print(f"Saved Structured Metadata: {metadata_path}")

# Verify round-trip loading
loaded_tensor = torch.load(cache_tensor_path, map_location="cpu")
assert torch.equal(loaded_tensor, activation_cpu), "Reloaded tensor does not match saved tensor!"
print("CHECK PASSED: CPU round-trip tensor persistence verified.")

## 10. Checkpointing & Google Colab Resilience Guide

Google Colab runtimes can disconnect or reset unexpectedly.  
To preserve activation caches and trained SAE checkpoints, use the utilities below.

In [ ]:
import shutil

def mount_google_drive(mount_point="/content/drive"):
    """Mounts Google Drive inside a Google Colab session."""
    if "google.colab" in sys.modules:
        from google.colab import drive
        drive.mount(mount_point)
        print(f"Google Drive successfully mounted at: {mount_point}")
    else:
        print("Local environment detected: skipping Google Drive mounting.")

def backup_checkpoints_to_drive(dest_folder="/content/drive/MyDrive/sae_gpt2_checkpoints"):
    """Backs up all checkpoints to Google Drive."""
    if os.path.exists("checkpoints"):
        os.makedirs(dest_folder, exist_ok=True)
        shutil.copytree("checkpoints", os.path.join(dest_folder, "checkpoints"), dirs_exist_ok=True)
        print(f"Checkpoints backed up to: {dest_folder}")
    else:
        print("No local 'checkpoints/' directory found to back up.")

def create_and_download_archive(archive_name="sae_phase1_checkpoints", target_dir="checkpoints"):
    """Compresses target directory and initiates browser download in Colab."""
    zip_path = shutil.make_archive(archive_name, "zip", target_dir)
    print(f"Created compressed archive: {zip_path} ({os.path.getsize(zip_path)} bytes)")
    if "google.colab" in sys.modules:
        from google.colab import files
        files.download(zip_path)
        print("Browser download prompt triggered.")
    else:
        print(f"Local environment: Archive saved at {zip_path}.")

print("Colab backup utilities defined successfully.")

## 11. Summary & Transition to Phase 2

Phase 1 has established all foundations:
- Verified dependencies and hardware detection.
- Pretrained `gpt2-small` loaded, in `eval()` mode, 100% frozen.
- Hook site `blocks.6.hook_resid_post` verified with gradient-free caching.
- CPU-compatible persistence and metadata schema verified.

**Proceed to `notebooks/01_sae_experiment.ipynb` for Phases 2 through 5:**
1. Extracting real token activations across WikiText-2 splits (`train`, `val`, `test`).
2. Training two SAE runs with contrasting sparsity penalties $\lambda$.
3. Comparative evaluation on the held-out evaluation set.